In [ ]:
# | default_exp parser.metadata


# Parser Metadata
> Utilities for extracting frontmatter metadata from markdown and Jupyter notebook files.

In [ ]:
#| export
import yaml
import json
from pathlib import Path
from bs4 import BeautifulSoup


## Metadata Parsing

In [ ]:
# | export
def parse_metadata(content: str  # Raw markdown content with YAML frontmatter
                   ) -> dict:
    "Extract metadata from YAML frontmatter."
    yaml_section = content.split("---")[1]
    metadata = yaml.safe_load(yaml_section)
    if "title" not in metadata and "pagetitle" in metadata:
        metadata["title"] = metadata.get("pagetitle", "")
    return metadata

In [ ]:
# | export
def parse_notebook_metadata(content: str  # Raw Jupyter notebook JSON string
                            ) -> dict:
    "Extract metadata from the first cell of a Jupyter notebook."
    notebook = json.loads(content)
    if notebook.get("cells"):
        first_cell = notebook["cells"][0]
        if first_cell.get("cell_type") == "markdown":
            source = "".join(first_cell.get("source", []))
            if source.startswith("---"):
                return parse_metadata(source)
    return {}


In [ ]:
# | test
from fastcore.test import test_eq
from pathlib import Path

sample_dir = Path("sample")
if not sample_dir.exists():
    sample_dir = Path("../sample")

with open(sample_dir / "example.md") as file:
    content = file.read()

metadata = parse_metadata(content)

with open(sample_dir / "design_questions.ipynb") as f:
    nb_content = f.read()
nb_metadata = parse_notebook_metadata(nb_content)
print(nb_metadata)


{}


## Notebook Parsing

Helpers for filtering and extracting content from Jupyter notebook cells.

In [ ]:
#| export
def is_frontmatter(cell: dict  # Notebook cell dict
                   ) -> bool:
    "Check if a cell is a YAML frontmatter cell."
    if cell.get("cell_type") == "markdown":
        return "".join(cell.get("source", [])).startswith("---")
    return False

In [ ]:
#| export
def is_visible_code(cell: dict,  # Notebook cell dict
                    is_quarto: bool = False  # Whether the notebook is a Quarto doc
                    ) -> bool:
    "Check if a code cell should be included in output."
    if cell.get("cell_type") != "code":
        return False
    if not is_quarto:
        return True
    cell_text = "".join(cell.get("source", []))
    hidden = "#| echo: false" in cell_text or "#| include: false" in cell_text
    return not hidden


In [ ]:
#| export
def extract_notebook_content(content: str,  # Raw Jupyter notebook JSON string
                             is_quarto: bool = False  # Whether the notebook is a Quarto doc
                             ) -> str:
    "Extract visible markdown and code content from a Jupyter notebook."
    cells = json.loads(content).get("cells", [])
    cells = filter(lambda c: not is_frontmatter(c), cells)
    cells = filter(lambda c: is_visible_code(c, is_quarto) or c.get("cell_type") == "markdown", cells)
    return "\n".join("".join(c.get("source", [])) for c in cells)

In [ ]:
# | export
def remove_metadata(content: str) -> str:
    """Remove frontmatter from content"""
    end = content.find("---", 3)
    return content[end + 3:].strip() if end != -1 else content


## HTML Extraction

In [ ]:
#| export
def extract_html_metadata(html: str) -> dict:
    "Extract SEO metadata from HTML meta tags."
    soup = BeautifulSoup(html, "html.parser")

    def meta(name=None, prop=None):
        if prop:
            tag = soup.find("meta", property=prop)
        else:
            tag = soup.find("meta", attrs={"name": name})
        return tag["content"].strip() if tag and tag.get("content") else ""

    return {
        "title": (soup.title.string.strip() if soup.title else "") or meta(prop="og:title"),
        "description": meta(name="description") or meta(prop="og:description"),
        "date": meta(prop="article:published_time")[:10] if meta(prop="article:published_time") else "",
    }
